In [ ]:
#| default_exp training_pipeline
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
from tqdm.auto import tqdm
import os
import joblib

# Imports from our previous modules
from adia_cybernet.core_data_processing import ECADataGenerator, SeriesProcessor, PermutationSymbolizer
from adia_cybernet.core_model_architecture import MDL_AU_Net_Autoencoder, StructuralBreakClassifier, HierarchicalArgs

# Introduction
This module contains the high-level orchestration logic for training our model. Our strategy involves a sophisticated two-stage training regimen designed to build a robust "Financial Seismologist":
1. MDL Pre-training (The "School"): We first train our MDL_AU_Net_Autoencoder on a vast, synthetic dataset of Elementary Cellular Automata (ECA) dynamics. The model learns the fundamental "physics" of rule-based systems by performing two tasks simultaneously: reconstructing the system's evolution and classifying the underlying rule from a compressed "fingerprint." This is managed by the MDLPreTrainer class.
1. Fine-tuning (The "On-the-Job Training"): We then take the pre-trained, "educated" encoder and adapt it for the specific task of detecting structural breaks in financial data. We pair it with a new classification head to form a StructuralBreakClassifier. This model is then fine-tuned on the real ADIA training data. This stage is managed by the BreakClassifierFinetuner class.
Finally, the train() function orchestrates this entire process, saving the final, fine-tuned encoder—our ready-to-use artifact for inference.

---

# Section 1: The MDL Pre-Trainer
This class encapsulates all the logic for Stage 1: pre-training our U-Net autoencoder on synthetic ECA data.

In [ ]:
#| export
class MDLPreTrainer:
    """Orchestrates the MDL pre-training stage on synthetic ECA data."""
    def __init__(self, model: MDL_AU_Net_Autoencoder, config: dict):
        self.model = model
        self.config = config
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=config['pretrain_lr'])
        self.recon_criterion = nn.BCEWithLogitsLoss() # Good for binary reconstruction
        self.class_criterion = nn.CrossEntropyLoss()
        self.device = config.get('device', 'cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)

    def pretrain(self, data_generator: ECADataGenerator) -> MDL_AU_Net_Autoencoder:
        """Runs the pre-training loop and returns the trained model."""
        print("--- Starting Stage 1: MDL Pre-training ---")
        self.model.train()

        # Generate the synthetic dataset
        X, y = data_generator.generate_training_data()
        dataset = TensorDataset(torch.from_numpy(X).float(), torch.from_numpy(y).long())
        loader = DataLoader(dataset, batch_size=self.config['pretrain_batch_size'], shuffle=True)

        for epoch in range(self.config['pretrain_epochs']):
            epoch_loss = 0
            progress_bar = tqdm(loader, desc=f"Pre-train Epoch {epoch+1}/{self.config['pretrain_epochs']}")

            for sequences, labels in progress_bar:
                sequences, labels = sequences.to(self.device), labels.to(self.device)

                self.optimizer.zero_grad()

                recon, logits = self.model(sequences)

                loss_recon = self.recon_criterion(recon, sequences)
                loss_class = self.class_criterion(logits, labels)

                # The core MDL dual-loss objective
                total_loss = self.config['alpha_loss'] * loss_recon + \
                             self.config['beta_loss'] * loss_class

                total_loss.backward()
                self.optimizer.step()

                epoch_loss += total_loss.item()
                progress_bar.set_postfix({'loss': total_loss.item()})

            print(f"Pre-train Epoch {epoch+1} Average Loss: {epoch_loss / len(loader):.4f}")

        print("--- MDL Pre-training Complete ---")
        return self.model

---

# Section 2: The Fine-Tuner
This class handles Stage 2: adapting the pre-trained encoder to the real-world ADIA dataset.

In [ ]:
#| export
class BreakClassifierFinetuner:
    """Orchestrates the fine-tuning stage on the real ADIA dataset."""
    def __init__(self, model: StructuralBreakClassifier, config: dict):
        self.model = model
        self.config = config
        # Fine-tune only the new classifier head and maybe the last layer of the encoder
        self.optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, self.model.parameters()), lr=config['finetune_lr'])
        self.criterion = nn.BCEWithLogitsLoss()
        self.device = config.get('device', 'cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)

    def finetune(self, X_train_df: pd.DataFrame, y_train: pd.Series, series_processor: SeriesProcessor) -> StructuralBreakClassifier:
        """Runs the fine-tuning loop."""
        print("\n--- Starting Stage 2: Fine-tuning for Structural Breaks ---")
        self.model.train()

        ids = y_train.index

        for epoch in range(self.config['finetune_epochs']):
            epoch_loss = 0
            # Shuffle IDs each epoch for stochasticity
            shuffled_ids = ids.to_series().sample(frac=1).index
            progress_bar = tqdm(shuffled_ids, desc=f"Finetune Epoch {epoch+1}/{self.config['finetune_epochs']}")

            for series_id in progress_bar:
                self.optimizer.zero_grad()

                series_df = X_train_df.loc[series_id]
                label_tensor = torch.tensor([y_train.loc[series_id]], dtype=torch.float32).to(self.device)

                before_series = series_df[series_df['period'] == 0]['value']
                after_series = series_df[series_df['period'] == 1]['value']

                # Process series into tensors
                processed_before = series_processor.process(before_series)
                processed_after = series_processor.process(after_series)

                # Skip if a segment is too short to process
                if processed_before is None or processed_after is None:
                    continue

                processed_before = processed_before.to(self.device)
                processed_after = processed_after.to(self.device)

                logit = self.model(processed_before, processed_after)
                loss = self.criterion(logit, label_tensor)

                loss.backward()
                self.optimizer.step()

                epoch_loss += loss.item()
                progress_bar.set_postfix({'loss': loss.item()})

            print(f"Finetune Epoch {epoch+1} Average Loss: {epoch_loss / len(ids):.4f}")

        print("--- Fine-tuning Complete ---")
        return self.model

---

# Section 3: The Artifact Saver
A simple utility function to save the final, trained encoder and its configuration, which are the only artifacts needed for inference.

In [ ]:
#| export
class EncoderSaver:
    """Saves the final trained encoder and its configuration."""
    def save(self, model: StructuralBreakClassifier, model_args: HierarchicalArgs, path: str):
        """Saves the encoder state dict and the model configuration."""
        if not os.path.exists(path):
            os.makedirs(path)

        encoder_path = os.path.join(path, "final_encoder.pth")
        config_path = os.path.join(path, "model_config.joblib")

        # Save only the state dictionary of the encoder sub-module
        torch.save(model.encoder.state_dict(), encoder_path)
        # Save the configuration dataclass needed to reconstruct the model
        joblib.dump(model_args, config_path)

        print(f"\n✅ Final encoder and config saved to '{path}'")

---

# Section 4: The Main train() Entrypoint
This function ties everything together. It defines the configurations, instantiates the components, and runs the full two-stage pipeline. This is the function the ADIA platform will call.

In [ ]:
#| export
def train(X_train_df: pd.DataFrame, y_train_series: pd.Series, model_directory_path: str):
    """
    Main entrypoint for the training process.
    Orchestrates pre-training, fine-tuning, and saving the final model artifact.
    """
    # --- 1. Define All Configurations ---
    training_config = {
        'pretrain_lr': 1e-4,
        'finetune_lr': 1e-5,
        'pretrain_batch_size': 32,
        'pretrain_epochs': 5, # In a real run, this would be much higher
        'finetune_epochs': 3,   # In a real run, this would be higher
        'alpha_loss': 1.0,    # Weight for reconstruction loss
        'beta_loss': 0.5,     # Weight for classification loss
        'device': 'cuda' if torch.cuda.is_available() else 'cpu'
    }

    # Model architecture configuration
    model_args = HierarchicalArgs(
        input_dim=6, # vocab_size from PermutationSymbolizer (3!)
        num_classes=4, # 2 base rules + 2 composite rules for our test
        dimensions=[64, 128], # A smaller model for faster training
        layers=[2, 2],
        max_seqlens=[64, 16],
        n_heads=4
    )

    # Data generation configuration
    eca_config = {
        "rules_to_use": {
            "base": [90, 110],
            "composite": {3: [90, 110]}
        },
        "width": model_args.input_dim,
        "steps": model_args.max_seqlens[0] * 20, # Generate enough data for windowing
        "n_reps": 50, # Number of simulations per rule
        "warmup": 100,
        "seed": 42
    }

    # Data processing configuration
    series_proc_config = {
        "embedding_dim": 3, # Must match vocab_size calculation for input_dim
        "sequence_length": model_args.max_seqlens[0]
    }

    # --- 2. Instantiate Components ---
    # Models
    autoencoder = MDL_AU_Net_Autoencoder(model_args)
    # The finetuning model uses the encoder from the autoencoder
    finetune_classifier = StructuralBreakClassifier(autoencoder.encoder, model_args.latent_dim)

    # Data Generators/Processors
    data_generator = ECADataGenerator(eca_config)
    symbolizer = PermutationSymbolizer(embedding_dim=series_proc_config['embedding_dim'])
    series_processor = SeriesProcessor(symbolizer, sequence_length=series_proc_config['sequence_length'])

    # Trainers
    pre_trainer = MDLPreTrainer(autoencoder, training_config)
    fine_tuner = BreakClassifierFinetuner(finetune_classifier, training_config)
    saver = EncoderSaver()

    # --- 3. Run the Pipeline ---
    # Stage 1
    pre_trainer.pretrain(data_generator)

    # Stage 2
    fine_tuner.finetune(X_train_df, y_train_series, series_processor)

    # Stage 3
    saver.save(finetune_classifier, model_args, model_directory_path)

---

# Section 5: Unit Tests
A "smoke test" to ensure the entire train function can execute end-to-end without crashing. It uses minimal data and epochs for speed.

In [ ]:
#| include: false
import tempfile
import shutil
import math

# --- Create a mock/dummy dataset for testing ---
def create_dummy_data(num_series=4, len_before=200, len_after=100):
    ids = [f"id_{i}" for i in range(num_series)]
    y_train = pd.Series([i % 2 == 0 for i in range(num_series)], index=ids, name="structural_breakpoint")

    all_series_dfs = []
    for i, series_id in enumerate(ids):
        # Create 'before' segment
        before_df = pd.DataFrame({
            'value': np.random.randn(len_before),
            'period': 0
        })
        before_df.index.name = 'id'
        before_df = pd.concat([before_df], keys=[series_id], names=['id'])

        # Create 'after' segment
        after_df = pd.DataFrame({
            'value': np.random.randn(len_after) + (0.5 if y_train[i] else 0), # Add shift if break
            'period': 1
        })
        after_df.index.name = 'id'
        after_df = pd.concat([after_df], keys=[series_id], names=['id'])

        all_series_dfs.extend([before_df, after_df])

    X_train = pd.concat(all_series_dfs)
    return X_train, y_train

# --- Define the Smoke Test ---
def test_train_pipeline_smoke_test():
    with tempfile.TemporaryDirectory() as temp_dir:
        print(f"Running smoke test, artifacts will be saved to: {temp_dir}")

        # Create tiny dummy data
        X_dummy, y_dummy = create_dummy_data()

        # Run the training function
        try:
            train(X_dummy, y_dummy, temp_dir)
        except Exception as e:
            assert False, f"The train() function crashed with an exception: {e}"

        # Verify that the final artifacts were created
        expected_encoder_path = os.path.join(temp_dir, "final_encoder.pth")
        expected_config_path = os.path.join(temp_dir, "model_config.joblib")

        assert os.path.exists(expected_encoder_path), "The encoder .pth file was not created."
        assert os.path.exists(expected_config_path), "The model config .joblib file was not created."
        print("\nSmoke test passed: train() ran without errors and created the expected artifacts.")

# --- Run the Test ---
test_train_pipeline_smoke_test()